# YSDA RecSys Course

## Двухбашенные модели. Домашнее задание

### ФИО: <Метялков Станислав Ильич>

В этом домашнем задании вы пройдёте полный базовый пайплайн: подготовка данных → метрики → несколько рекомендательных подходов → итоговый лидерборд. В качестве рекомендательных подходов вы реализуете различные лосс-функции, которые используются для обучения двухбашенных моделей для стадии отбора кандидатов.

### Данные
Данные лежат в архиве `data.zip`, который состоит из:
* `interactions.parquet` - user-item взаимодействия из датасета Yambda (лайки для 500m версии)
* `embeddings.parquet` - уже пофильтрованные и чуть более плотно запакованные эмбеддинги треков из Yambda
* `artists.parquet` - метаданные айтемов с маппингом в артистов

Скачать архив можно здесь: [ссылка на google disk](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing). В следующем блоке мы в любом случае скачиваем датасет, поэтому самостоятельно его можно не качать.

### Guidelines
- Для выполнения ДЗ достаточно использовать Google Collab с T4
- Детерминизм: фиксируйте сиды там, где это важно
- Не используйте данные из теста при обучении/подготовке моделей
- Тест — **последняя неделя** (по timestamp), как описано ниже
- После каждого этапа запускайте проверки внутри ноутбука
- Старайтесь избегать работы с сырыми питоновскими объектами (словарями, списками, интами) там, где можно применить методы из `polars` - они будут в десятки-сотни раз быстрее и читабельней
- Чтобы быстрее обнаруживать, что код написан неоптимально и выполняется слишком долго, старайтесь оборачивать циклы в `tqdm` и выводить progress bar
- Во время отладки кода можно посэмплировать данные для скорости с помощью `data.sample(fraction=0.1, seed=42)`. Для проверки тестов после отладки надо сделать запуски с полным датасетом

### Разбалловка
Основные задания:
1) Подготовка данных, оценка качества, создание датасета и коллатора для обучения - 1 балл
2) Реализация графа вычислений для обучения двухбашенной модели (без лосса) и инференса (получения кандидатов) - 1 балл
3) Цикл обучения - 1 балл
4) Softmax loss - 1 балл
5) BCE loss - 1 балл
6) BPR loss - 1 балл
7) Sampled softmax, uniform negatives - 1 балл
8) Sampled softmax, in-batch negatives - 1 балл
9) Sampled softmax, in-batch negatives + logq correction - 1 балл
10) Ответы на вопросы в конце ноутбука - 1 балл

Бонусные задания:
1) Sampled softmax, in-batch negatives + **fixed** logq correction - 1 балл
2) Улучшение аггрегации история пользователя - 1 балл

In [ ]:
!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -oq dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=6816a0bd-c072-4025-910a-44caecbcd7d5
To: /content/dataset.zip
100% 356M/356M [00:02<00:00, 147MB/s]


In [ ]:
from collections import defaultdict
import copy
import gc
import os
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import polars as pl
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import tests

# 1. Подготовка данных, оценка качества, создание датасета и коллатора для обучения - 1 балл

## Подготовка данных

**Задача:**
1) Считать данные (взаимодействия, эмбеддинги, метаданные).  
2) Оставить только взаимодействия, для которых есть эмбеддинги.  
3) Сделать core фильтрацию: оставить только айтемы с ≥5 взаимодействиями (в ДЗ мы делаем это исключительно для удобства и скорости, в реальной работе так делать не стоит)
4) Поджоинить метаданные (артисты треков) ко всем взаимодействиям. Ремарка: здесь у нас если у одного трека несколько артистов произойдет дублирование прослушивание - в рамках ДЗ мы ничего с этим не делаем, но в реальной жизни такого допускать нельзя.
5) Сделать train-test split: последнюю неделю положить в тест.  
6) Ограничить тест юзерами, у которых есть взаимодействия в трейне.  
7) Подготовить для оценки качества `test_targets: Dict[uid, List[item_id]]`.
8) Оставить только эмбеддинги для core айтемов (остальные пофильтровать).

После этого блока должны существовать `train`, `test`, `embeddings`, `artists`, `test_targets`.

Для этого блока полезны как минимум следующие методы:
* `pl.read_parquet` - для чтения данных
* `.filter, .value_counts` помогут сделать core-фильтрацию
* `df.join(...)` - при джойне метаданных надо использовать `how='left'`, а не `how='inner'`
* `df.join(other, on=some_key, how='semi')` - режим `semi` используется для фильтраций (оставить только те строки из исходного датафрейма, ключ которых присутствует во второй таблице)

In [ ]:
# Пути к данным (ожидается, что они лежат рядом с ноутбуком)
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# Глобальные параметры
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# Для воспроизводимости
np.random.seed(42)

data = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)

print(f"embs: {len(embeddings):,}")
data = data.join(embeddings.select("item_id"), on="item_id", how="semi")

item_counts = data.group_by("item_id").agg(pl.len().alias("count"))
core_items = item_counts.filter(pl.col("count") >= CORE_MIN_INTERACTIONS_PER_ITEM).select("item_id")

data = data.join(core_items, on="item_id", how="semi")

data = data.join(artists, on="item_id", how="left")
max_timestamp = data.select(pl.col("timestamp").max()).item()

test_timestamp = max_timestamp - TEST_INTERVAL_SECONDS
test = data.filter(pl.col("timestamp") >= test_timestamp)
train = data.filter(pl.col("timestamp") < test_timestamp)

train_users = train.select("uid").unique()
test = test.join(train_users, on="uid", how="semi")


test_grouped = test.group_by("uid").agg(pl.col("item_id"))
test_targets = dict(zip(test_grouped["uid"].to_list(),test_grouped["item_id"].to_list()))
embeddings = embeddings.join(core_items, on="item_id", how="semi")

print(f"targs: {len(test_targets):,}")

unique_items = train.select("item_id").unique().sort("item_id")
item_to_idx = dict(zip(
    unique_items["item_id"].to_list(),
    range(len(unique_items))
))
idx_to_item = {v: k for k, v in item_to_idx.items()}

train = train.with_columns(
    pl.col("item_id").replace(item_to_idx).alias("item_idx")
)

test = test.with_columns(
    pl.col("item_id").replace(item_to_idx).alias("item_idx")
).filter(pl.col("item_idx").is_not_null())

catalog_size = len(item_to_idx)

embs: 591,811
targs: 37,446


In [ ]:
tests.check_data_split(train=train, test=test, embeddings=embeddings, artists=artists, test_targets=test_targets)

All good! :)


## Оценка качества

#### Определения метрик

Пусть для пользователя $u$:

* $G_u \subset \mathcal{I}$ — множество релевантных айтемов (ground truth)
* $R_u = (r_{u,1}, \dots, r_{u,K})$ — упорядоченный список рекомендаций длины $K$

Обозначим индикатор релевантности $I_{u,k} = [ r_{u, k} \in G_u]$. В простонародье его еще часто называют `hits`.

**Hitrate@K** равен единичке, если мы угадали в topK хотя бы один релевантный айтем:
* $
\text{Hitrate@K} = \frac{1}{|U|}
\sum_{u \in U}
\left[ \sum_{k=1}^{K} I_{u,k} > 0 \right]
$

**Recall@K** оценивает долю угаданных релевантных айтемов (от всех релевантных айтемов):
* $
\text{Recall@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{
\sum_{k=1}^{K} I_{u,k}
}{
\min(|G_u|, K)
}
$

Для подсчета **nDCG@K** нужно сначала посчитать **DCG@K**, затем посчитать **iDCG@K** (DCG в случае идеального ранжирования), затем одно поделить на другое:
* $
\text{DCG@K}(u) = \sum_{k=1}^{K}
\frac{I_{u,k}}{\log_2(k+1)}
$
* $
\text{iDCG@K}(u) = \sum_{k=1}^{\min(|G_u|,K)}
\frac{1}{\log_2(k+1)}
$
* $
\text{nDCG@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{\text{DCG@K}(u)}{\text{iDCG@K}(u)}
$

**Coverage@K** - это число уникальных айтемов во всех рекомендациях, деленное на размер каталога:

* $
\text{Coverage@K} = \frac{|\bigcup_{u \in U} R_u|}{|\mathcal{I}_{train}|},
$ где $\mathcal{I}_{train}$ — каталог айтемов в train.
* в качестве размера каталога используем количество айтемов, которые нам доступны для рекомендации на момент рекомендации (то есть количество уникальных айтемов в `train`)

#### Что нужно сделать
Реализуйте функции:
- `get_metrics(targets, candidates, topk) -> dict(hitrate, recall, ndcg)`
- `evaluate(targets_by_user, candidates_by_user, catalog_size, topk) -> dict(hitrate, recall, ndcg, coverage)`

**Важно:**
* `candidates[uid]` должен иметь длину ровно `topk`
* при подсчете метрики нужно дедуплицировать позитивы; это влияет на значение, на которое мы делим


In [ ]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:

    uniq_targets = set(targets)
    num_relevant = len(uniq_targets)

    hits = [1 if c in uniq_targets else 0 for c in candidates[:topk]]
    sum_hits = sum(hits)
    hitrate = 1 if sum_hits > 0 else 0



    dcg = sum(hits[i] / np.log2(i + 2) for i in range(len(hits)))

    ideal_len = min(num_relevant, topk)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_len))

    ndcg = dcg / idcg if idcg > 0 else 0

    recall = sum_hits / min(num_relevant, topk) if num_relevant > 0 else 0

    return {
        "hitrate": hitrate,
        "ndcg": ndcg,
        "recall": recall,
    }


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:

    hitrates = []
    recalls = []
    ndcgs = []
    all_recommended_items = set()

    for uid in targets:
        user_targets = targets[uid]
        user_candidates = candidates[uid]

        metrics = get_metrics(user_targets, user_candidates, topk)
        hitrates.append(metrics["hitrate"])
        recalls.append(metrics["recall"])
        ndcgs.append(metrics["ndcg"])


        all_recommended_items.update(user_candidates[:topk])
    coverage = len(all_recommended_items) / catalog_size

    hitrate = np.mean(hitrates)
    ndcgs = np.mean(ndcgs)
    recall = np.mean(recalls)

    return {
        "hitrate": hitrate,
        "ndcg": ndcgs,
        "coverage": coverage,
        "recall": recall,
    }

In [ ]:
tests.check_metrics(get_metrics=get_metrics, evaluate=evaluate)

All good! :)


## Создание датасета и коллатора для обучения

Вам нужно реализовать несколько вспомогательных функций для работы с пользовательскими историями переменной длины. Эти функции будут использоваться дальше при построении семплов, батчей и обучении моделей, поэтому важно сразу договориться о формате представления последовательностей.

#### Формат данных: flatten-представление истории

Вместо того чтобы хранить историю каждого пользователя как отдельный массив (и затем делать padding до общей длины), мы будем хранить всю историю батча в одном “плоском” тензоре — в формате flatten:

`[u1_t1, u1_t2, ..., u1_tL1, u2_t1, ..., u2_tL2, ...]`

То есть в одном тензоре подряд записаны взаимодействия пользователя 1, затем пользователя 2, и так далее.

Чтобы при этом не потерять границы между пользователями, отдельно хранится тензор `length`, где указано, сколько элементов истории относится к каждому пользователю в батче:

`length = [L1, L2, ..., LB]`

где `B` — размер батча, а `Li` — длина истории `i`-го пользователя.


В таком формате будет храниться именно пользовательская история(последовательность взаимодействий), а ваша задача — реализовать функции, которые позволяют:
- восстанавливать границы последовательностей по `length`,
- превращать flatten-представление в padded-формат + mask,
- готовить батч к подаче в модель.

### Функция `create_masted_tensor`

Напишите функцию `create_masked_tensor`, которая по `flatten` представлению батча последовательностей и их длинам формирует `padded` тензор и булеву маску позиций элементов.

In [ ]:
def create_masked_tensor(data: torch.Tensor, lengths: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
  """
  Converts a batch of flattened variable-length sequences into a padded tensor and mask.
  Supports:
    - indices: data shape (total_num_elements,)
    - embeddings/features: data shape (total_num_elements, d1, d2, ...)

  Parameters
  ----------
  data : torch.Tensor
      Input tensor containing flattened sequences:
      - For indices: shape (total_num_elements,)
      - For embeddings: shape (total_num_elements, embedding_dim)
  lengths : torch.Tensor
      1D tensor of sequence lengths, shape (batch_size,). Specifies the actual length
      of each sequence.

  Returns
  -------
  Tuple[torch.Tensor, torch.Tensor]
      - padded_tensor: Padded tensor of shape:
          - (batch_size, max_seq_len) for indices
          - (batch_size, max_seq_len, embedding_dim) for embeddings
          Shorter sequences are right-padded with zeros.
      - mask: Boolean mask of shape (batch_size, max_seq_len) where True indicates
          valid elements and False indicates padding. Can be used in attention or loss computation.

  Examples
  --------
  >>> data = torch.tensor([1, 2, 3, 4, 5, 6])  # sequences: [1,2], [3,4,5], [6]
  >>> lengths = torch.tensor([2, 3, 1])
  >>> padded, mask = create_masked_tensor(data, lengths)
  >>> padded
  tensor([[1, 2, 0],
          [3, 4, 5],
          [6, 0, 0]])
  >>> mask
  tensor([[ True,  True, False],
          [ True,  True,  True],
          [ True, False, False]])
  """
  bs = lengths.shape[0]
  max_seq_len = lengths.max().item()
  mask = torch.arange(max_seq_len, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)

  if data.dim() == 1:
      padded_shape = (bs, max_seq_len)
  else:
      padded_shape = (bs, max_seq_len) + data.shape[1:]
  padded = torch.zeros(padded_shape, dtype=data.dtype, device=data.device)

  padded[mask] = data

  return padded, mask

In [ ]:
tests.test_create_masked_tensor(create_masked_tensor)

All good! :)


### Класс `YambdaDataset`

Реализуйте класс `YambdaDataset`, который работает с пользовательскими историями взаимодействий и подготавливает семплы для последующего обучения моделей.

Датасет должен поддерживать два режима работы, задаваемые флагом `is_train`, а также обрезку истории до последних `max_seq_len` элементов.

В train-mode мы превращаем одну пользовательскую историю длины `T` в `T-1` обучающих семплов. То есть для пользователя с историей `[i1, i2, ..., iT]` мы создаём семплы с префиксами:

- `history[:1] -> label = i2`
- `history[:2] -> label = i3`
- ...
- `history[:T-1] -> label = iT`

Если реализовать это “в лоб” и в `__init__` материализовать все такие семплы, то мы получим сильное дублирование данных: один и тот же айтем `i1` будет повторяться почти во всех семплах, `i2` — во всех, кроме первого, и т.д.

Поэтому в `__init__` мы храним только индексы/указатели, а сам префикс от `history` и обрезку строим на лету в `__getitem__`.

#### Входные данные

- `histories: Dict[uid, List[int]]` — временно упорядоченные истории пользовательских взаимодействий.
- `labels: Dict[uid, List[int]]` — целевые айтемы пользователя для оценки (в нашем случае последняя неделя).
- `is_train: bool` — режим работы датасета.
- `max_seq_len: int` — максимальная длина возвращаемой истории (по умолчанию `100`).

#### Режим 1: Train mode (`is_train=True`)

В режиме обучения датасет должен подготовить семплы на пользователя в постановке next-item prediction.

Если история пользователя: `[i1, i2, ..., iT]`, то нужно создать `T - 1` семплов. Для каждого `t` от `1` до `T-1` (позиция следующего айтема):

- `history` = префикс `history[:t]`, обрезанный до последних `max_seq_len` элементов
- `label` = следующий айтем `history[t]`

Формат train-семпла:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  },
  "label": int
}
```

#### Режим 2: Inference mode (`is_train=False`)

В режиме оценки датасет должен возвращать ровно один семпл на пользователя.
Пользователь попадает в датасет только если для него есть таргеты в `labels`.
Содержимое семпла:
- `history` = история пользователя, обрезанная до последних `max_seq_len` элементов.

Формат eval-семпла:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  }
}
```

In [ ]:
class YambdaDataset(Dataset):
  """
  PyTorch Dataset for user interaction histories with next-item prediction samples.

  Parameters
  ----------
  histories : Dict[Any, List[int]]
      Mapping from user id to a list of interacted item ids (sorted by time).
  labels : Dict[Any, List[int]]
      Mapping from user id to a list of target item ids.
      Used only to filter users in eval mode (`uid in labels`).
  is_train : bool
      If True, generate multiple (prefix, next_item) samples per user.
      If False, return one sample per user (filtered by presence in `labels`).
  max_seq_len : int, default 100
      Maximum number of most recent items to keep in the returned history.

  Returns
  -------
  Dict[str, Any]
      Train mode (`is_train=True`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
            "label": int,
          }

      Eval mode (`is_train=False`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
          }

      where:
        - history["item_id"] contains up to `max_seq_len` last items of the selected prefix/history
        - history["length"] is the length of the returned (possibly truncated) history
        - label is a single next item id (int)

  Examples
  --------
  Train mode:
  >>> ds = YambdaDataset(histories, labels={}, is_train=True, max_seq_len=100)
  >>> s = ds[0]
  >>> s["uid"]
  >>> s["history"]["item_id"], s["history"]["length"]
  >>> s["label"]

  Eval mode (filters users by `labels` keys):
  >>> ds = YambdaDataset(histories, labels=test_targets, is_train=False)
  >>> s = ds[0]
  >>> s["uid"]
  >>> s["history"]["item_id"], s["history"]["length"]
  """

  def __init__(
        self,
        histories: Dict[Any, List[int]],
        labels: Dict[Any, List[int]],
        is_train: bool,
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()
        self.histories = histories
        self.labels = labels
        self.is_train = is_train
        self.max_seq_len = max_seq_len

        if is_train:
            self.samples = []
            for uid, history in histories.items():
                for t in range(1, len(history)):
                    self.samples.append((uid, t))
        else:
            self.samples = [uid for uid in histories.keys() if uid in labels]

  def __len__(self) -> int:
      """Return number of samples (prefix samples in train mode, users in eval mode)."""
      return len(self.samples)

  def __getitem__(self, idx: int) -> Dict[str, Any]:
        """
        Build and return a single sample using an index pointer (uid, t).
        """
        if self.is_train:
            uid, t = self.samples[idx]
            history = self.histories[uid]
            prefix = history[:t]

            if len(prefix) > self.max_seq_len:
                prefix = prefix[-self.max_seq_len:]

            label = history[t]

            return {
                "uid": uid,
                "history": {
                    "item_id": prefix,
                    "length": len(prefix)
                },
                "label": label
            }
        else:
            uid = self.samples[idx]
            history = self.histories[uid]

            if len(history) > self.max_seq_len:
                history = history[-self.max_seq_len:]

            return {
                "uid": uid,
                "history": {
                    "item_id": history,
                    "length": len(history)
                }
            }

In [ ]:
tests.test_yambda_dataset(YambdaDataset)

All good! :)


### Функция `collate_fn`

Реализуйте функцию `collate_fn`, которая будет использоваться в `DataLoader` для преобразования списка семплов
из `YambdaDataset` в батчи, удобные для подачи в модель и работы с ними.

Как говорилось ранее, мы используем flatten-представление: вместо padding до общей длины мы
1) конкатенируем все пользовательские истории в батче в один 1D-тензор  
2) отдельно сохраняем `length`, чтобы позже восстановить границы последовательностей


#### Вход

`batch: List[Dict[str, Any]]` — список семплов из `YambdaDataset`.

#### Что должна сделать `collate_fn`

Функция должна сформировать единый словарь, где все значения — `torch.Tensor` типа `torch.long`.

- `result["history"]["item_id"]` 1D тензор, полученный конкатенацией всех `history["item_id"]` в порядке объектов в `batch` размерность: `(sum(history_lengths),)`

- `result["history"]["length"]` 1D тензор длин историй для каждого объекта батча размерность: `(batch_size,)`

- `result["uid"]` 1D тензор идентификаторов пользователей размерность: `(batch_size,)`

- `result["label"]` (только если во входных семплах есть `"label"`) 1D тензор лейблов (next item id) в порядке объектов батча размерность: `(batch_size,)`

#### Требования

- Не использовать `padding`. Только `flatten`-конкатенация + `lengths`.
- Сохранять порядок объектов в `batch` при конкатенации.
- Возвращать `"label"` только если он присутствует во входных семплах.
- Все числовые значения должны быть приведены к `torch.Tensor` типа `torch.long`.


In [ ]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
  """
  Collate function that converts a list of samples into a **flatten** batch representation.

  This function implements the "flatten" batching scheme: instead of padding variable-length
  sequences to a common length, it concatenates all user histories in the batch into a single
  1D tensor and returns a companion `length` tensor to recover per-user boundaries later.

  The function is compatible with `YambdaDataset` in two modes:
    - Train mode samples contain keys: `"uid"`, `"history"`, and `"label"` (where `"label"` is an `int`).
    - Eval mode samples contain keys: `"uid"` and `"history"`.

  Output batch format
  -------------------
  The returned dictionary contains:
    - `result["history"]["item_id"]`: 1D tensor with all history items concatenated in the
      order of samples in `batch`, shape `(sum(history_lengths),)`, dtype `torch.long`.
    - `result["history"]["length"]`: 1D tensor of per-sample history lengths,
      shape `(batch_size,)`, dtype `torch.long`.
    - `result["uid"]`: 1D tensor of user ids, shape `(batch_size,)`, dtype `torch.long`.
    - If `"label"` is present in the input samples (train batches):
        - `result["label"]`: 1D tensor of labels (next item ids), shape `(batch_size,)`,
          dtype `torch.long`.

  Parameters
  ----------
  batch : List[Dict[str, Any]]
      List of samples returned by the dataset `__getitem__`.

  Returns
  -------
  Dict[str, Any]
      A nested dictionary where all returned values are `torch.Tensor` objects.

  Examples
  --------
  - Train-mode: returns `"history"` + `"uid"` + `"label"` (1D tensor of next-item ids).
  - Eval-mode: returns `"history"` + `"uid"` (no `"label"` key).
  """
  uids = []
  all_item_ids = []
  lengths = []
  labels = []

  has_labels = "label" in batch[0]

  for sample in batch:
    history_items = sample["history"]["item_id"]
    all_item_ids.extend(history_items)
    lengths.append(sample["history"]["length"])
    uids.append(sample["uid"])

    if has_labels:
      labels.append(sample["label"])

  result = {
    "history": {
        "item_id": torch.tensor(all_item_ids, dtype=torch.long),
        "length": torch.tensor(lengths, dtype=torch.long)
    },
    "uid": torch.tensor(uids, dtype=torch.long)
  }

  if has_labels:
      result["label"] = torch.tensor(labels, dtype=torch.long)

  return result

In [ ]:
tests.test_collate_fn(collate_fn)

All good! :)


# 2. Реализация графа вычислений для обучения двухбашенной модели (без лосса) и инференса (получения кандидатов) (2 балла)

## UserEncoder

Реализуйте класс `UserEncoder`, который является основным компонентом нашей модели.

`UserEncoder` — это модуль, который по истории взаимодействий пользователя строит его контекстное представление.

Каждый пользователь $u$ описывается историей его взаимодействий $S_u$.

Каждому айтему $i$ из каталога соответствует обучаемый эмбеддинг $e_i \in \mathbb{R}^d$.

Представление для пользователя $u$, $P_u$, получается как агрегат эмбеддингов всех его предыдущих взаимодействий. В этом домашнем задании в качестве агрегации необходимо реализовать представление пользователя в духе
bag-of-words по его истории взаимодействий.

Для пользователя $u$ с историей взаимодействий $i_1, i_2, \ldots, i_{|S_u|}$ требуется получить:

$$
P_u = \sum_{k=1}^{|S_u|} e_{i_k}.
$$

#### Вход модели

Во время обучения данные из `YambdaDataset` с помощью `collate_fn` преобразуются в `flatten`-батчи и `batch["history"]` подается на вход метода `UserEncoder.forward` для получение представлений пользователей из батча.

#### Что должна сделать модель

1. Преобразовать полученные `item_id` в эмбеддинги объектов;
2. Посчитать представления пользователей в виде тензора размера `(batch_size, embedding_dim)`.
3. Вернуть полученные представления

In [ ]:
class UserEncoder(nn.Module):
  """
  User encoder that represents each user by a cumulative prefix sum of item embeddings.

  Parameters
  ----------
  num_items : int
      Total number of unique items in the catalog.
      Item ids must be in ``[0, num_items - 1]``.
  embedding_dim : int
      Dimension of item embeddings.

  Forward input
  -------------
  inputs : Dict[str, torch.Tensor]
      Dictionary with keys:
      - "item_id": Flattened item indices for concatenated sequences,
        shape ``(total_num_events,)``, dtype ``torch.long``.
      - "length": Per-user sequence lengths, shape ``(batch_size,)``,
        dtype ``torch.long``.

  Forward output
  --------------
  torch.Tensor
      User representations, one vector per user, shape ``(batch_size, embedding_dim)``.
  """
  def __init__(self, num_items: int, embedding_dim: int) -> None:
    super().__init__()
    self.item_embeddings = nn.Embedding(num_items, embedding_dim)

  def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
     embs = self.item_embeddings(inputs["item_id"])
     padded, mask = create_masked_tensor(embs, inputs["length"])
     user_repr = padded.sum(dim=1)
     return user_repr

In [ ]:
tests.test_user_encoder(UserEncoder)

All good! :)


## TwoTowerModel: обучение и инференс

`TwoTowerModel` объединяет `UserEncoder` и логику обучения/инференса модели.

#### Обозначения

$\mathbf{E} \in \mathbb{R}^{|I| \times d}$ — таблица эмбеддингов айтемов

$\mathbf{P}_u \in \mathbb{R}^d$ — представление пользователя $u$

Релевантность айтема $i$ для пользователя $u$: $r_i = \langle \mathbf{E}_{i}, \mathbf{P}_{u}\rangle$.


#### Что должна делать модель

Нам дан батч:
`inputs["history"]`: история (то, на основе чего строим пользователя и обучаетмся)
`inputs["labels"]`: таргеты/позитивы (айтемы с последней недели, по которым хотим получать метрики на эвале)

Для каждого пользователя в батче:
- строим $\mathbf{U}$ по его истории

#### Режим обучения (`self.training == True`)

- прогнать `inputs["history"]` через `UserEncoder` и получить $\mathbf{U}$ для пользователей в батче
- вычислить лосс через метод `compute_loss`
- вернуть лосс

#### Режим эвала (`self.training == False`):

- прогнать `inputs["history"]` через `UserEncoder` и получить $\mathbf{U}$ для пользователей в батче
- посчитать: $\text{all\_scores} = \langle\mathbf{U}, \mathbf{E}^{\top}\rangle$ размера `(batch_size, num_items)`
- вернуть тензор `all_scores` (метрики считаются отдельно)


#### Откуда берутся позитивы на обучении

Обучение формулируется как задача `next item prediction`. Для каждого шага в пользовательской истории позитивным примером считается следующий айтем в последовательности пользователя. Иными словами, модель обучается предсказывать следующий объект взаимодействия на основе всех предыдущих.
    
    

In [ ]:
class TwoTower(nn.Module):
  """
  Recommendation model combining user encoder with training and inference logic.

  The model produces:
    - a user representation vector `P_u` via `UserEncoder`
    - an item representation matrix `E` from the embedding table
    - uses dot-product relevance scores: `r_{ui} = <P_u, E_i>`.

  The `forward` method behaves differently depending on `self.training`:

  Training mode (`self.training == True`)
    - Encodes users.
    - Delegates loss computation to `compute_loss(...)`.
    - Returns a loss tensor.

  Evaluation / inference mode (`self.training == False`)
    - Encodes users.
    - Computes scores against all items in the catalog.
    - Returns a full score matrix.

  Parameters
  ----------
  num_items : int
    Total number of unique items in the catalog. Item ids must be in `[0, num_items - 1]`.
  embedding_dim : int
    Dimension of user/item embeddings.

  Notes
  -----
  This base class does not implement `compute_loss`. Subclasses should override it to define a training objective.
  """

  def __init__(self, num_items: int, embedding_dim: int) -> None:
    super().__init__()
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.init_weights(0.02)

  @torch.no_grad()
  def init_weights(self, initializer_range: float) -> None:
    """
    Initialize all model parameters with truncated normal distribution.

    Parameters
    ----------
    initializer_range : float
        Standard deviation of the truncated normal initializer.
    """
    for key, value in self.named_parameters():
      assert "weight" in key
      nn.init.trunc_normal_(
        value.data,
        std=initializer_range,
        a=-2 * initializer_range,
        b=2 * initializer_range,
      )

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    """
    Compute training loss.

    Parameters
    ----------
    user_repr : torch.Tensor
        User representations returned by the encoder, shape ``(batch_size, embedding_dim)``.
    inputs : Dict[str, Any]
        Full input batch. Expected to contain at least:
          - ``inputs["history"]``: dict with flattened history fields
          - label information (e.g., ``inputs["label"]``), depending on the training setup

    Returns
    -------
    torch.Tensor
        Scalar loss tensor.
    """
    # Эту функцию мы реализуем отдельно позже!
    # Не трогать ее и не менять здесь!
    raise NotImplementedError

  def forward(self, inputs: Dict[str, Any]) -> Dict[str, torch.Tensor]:
    """
    Run a forward pass with mode-dependent behavior.
    During training: computes and returns loss.
    During evaluation: computes and returns ranking scores for all items.

    Parameters
    ----------
    inputs : Dict[str, Any]
        Batch dictionary produced by `collate_fn`. Expected keys:
          - ``"history"``: dict with
                - ``"item_id"``: 1D flattened history item ids
                - ``"length"``: per-user history lengths
          - ``"uid"``: user ids tensor

    Returns
    -------
    torch.Tensor
        - If training (self.training == True): loss, scalar tensor
        - If evaluating (self.training == False): all_scores, relevance scores for all items with shape (batch_size, num_items)
    """
    user_repr = self.encoder(inputs["history"])
    if self.training:
      loss = self.compute_loss(user_repr, inputs)
      return loss

    else:
      item_embeddings = self.encoder.item_embeddings.weight
      all_scores = user_repr @ item_embeddings.T
      return all_scores

In [ ]:
tests.test_two_tower(TwoTower)

All good! :)


Посмотрим на то, что у нас получилось

In [ ]:
TRAIN_BATCH_SIZE = 2048
EVAL_BATCH_SIZE = 2048


train_histories = dict(
    train.group_by("uid")
    .agg(pl.col("item_idx").sort_by("timestamp"))
    .iter_rows()
)

test_targets = dict(
    test.group_by("uid")
    .agg(pl.col("item_idx"))
    .iter_rows()
)


yambda_train_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=True
)

yambda_eval_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=False
)

yambda_train_dataloader = DataLoader(
  dataset=yambda_train_dataset,
  batch_size=TRAIN_BATCH_SIZE,
  shuffle=True,
  collate_fn=collate_fn,
  drop_last=True
)

yambda_eval_dataloader = DataLoader(
  dataset=yambda_eval_dataset,
  batch_size=EVAL_BATCH_SIZE,
  shuffle=False,
  collate_fn=collate_fn,
  drop_last=False
)

# 3. Цикл обучения (1 балл)

Реализуйте функцию `evaluation`, которая выполняет оценку качества модели рекомендаций.

Функция должна:
1) Получить top-k рекомендаций для каждого пользователя из `dataloader`.
2) Собрать их в словарь формата `Dict[uid, List[item_id]]`.
3) Посчитать метрики, вызвав `evaluate(...)`, и вернуть результат.

#### Tips & Tricks
- Не забудьне перевести модель в `.eval()` режим
- Метрики можно считать с помощью функции `evaluate` из ДЗ 1

In [ ]:
def evaluation(
  dataloader: DataLoader,
  model: TwoTower,
  catalog_size: int,
  topk: int,
  device: str = "cuda",
) -> Dict[str, float]:
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################
  model.eval()
  model.to(device)
  candidates = {}

  with torch.no_grad():
    for batch in tqdm(dataloader, desc="Evaluation"):
      batch_device = {
          "history": {
              "item_id": batch["history"]["item_id"].to(device),
              "length": batch["history"]["length"].to(device)
          },
          "uid": batch["uid"].to(device)
      }

      all_scores = model(batch_device)
      nn, topk_indices = torch.topk(all_scores, k=topk, dim=1)
      uids = batch["uid"].tolist()
      topk_items = topk_indices.cpu().tolist()

      for uid, items in zip(uids, topk_items):
          candidates[uid] = items

  targets = dataloader.dataset.labels
  metrics = evaluate(
      targets=targets,
      candidates=candidates,
      catalog_size=catalog_size,
      topk=topk
  )

  return metrics


Реализуйте функцию `train`, которая обучает модель и после каждой эпохи запускает валидацию.

После завершения каждый эпохи необходимо:
- Запустить функцию `evaluation` на `valid_dataloader`
- Вывести метрики валидации в читаемом виде.
- Посчитать и вывести средний лосс за эпоху.

После окончания обучения вывести сообщение о завершении и вернуть состояние модели (`state dict`).

#### Примечания

- Важно корректно переключать режимы модели:
  - обучение выполняется в `train` режиме,
  - валидация должна выполняться внутри `evaluation`, где модель переводится в `eval` режим.
- Перенос батча на `device` должен корректно работать со структурой батча, где могут встречаться вложенные словари с тензорами.

In [ ]:
def train_model(
    train_dataloader: DataLoader,
    valid_dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    catalog_size: int,
    topk: int,
    device: str = "cuda"
  ) -> Dict[str, Any]:
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################
  model.to(device)

  for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1} of {num_epochs}"):
      batch_device = {
          "history": {
              "item_id": batch["history"]["item_id"].to(device),
              "length": batch["history"]["length"].to(device)
          },
          "uid": batch["uid"].to(device),
          "label": batch["label"].to(device)
      }

      optimizer.zero_grad()
      loss = model(batch_device)

      loss.backward()
      optimizer.step()

      total_loss += loss.item()
      num_batches += 1

    avg_loss = total_loss / num_batches
    print(f"\nEpoch {epoch + 1}/{num_epochs} - Average Train Loss: {avg_loss:.4f}")

    metrics = evaluation(
        dataloader=valid_dataloader,
        model=model,
        catalog_size=catalog_size,
        topk=topk,
        device=device
    )

    print(f"Metrics:")
    print(f"  Hitrate@{topk}: {metrics['hitrate']:.4f}")
    print(f"  Recall@{topk}:  {metrics['recall']:.4f}")
    print(f"  nDCG@{topk}:    {metrics['ndcg']:.4f}")
    print(f"  Coverage@{topk}: {metrics['coverage']:.4f}")
    print("-" * 80)

  print("Training finished")

  return model.state_dict()

# Реализуем различные способы обучения полученной двухбашенной модели

In [ ]:
# DO NOT CHANGE!
NUM_EPOCHS = 1
LEARNING_RATE = 1e-3
DEVICE = "cuda"

## 4. Softmax loss (1 балл)



В задаче отбора кандидатов каждому пользователю и каждому айтему сопоставляется векторное представление размерности $d$ в общем латентном пространстве.

Скор релевантности пользователя $u$ и айтема $i$ вычисляется как скалярное произведение их эмбеддингов:

$$
r(u, i) = \langle \mathbf{E}_i,\;\mathbf{P}_u \rangle,
$$

где:
- $\mathbf{P}_u \in \mathbb{R}^d$ — представление пользователя, полученное из `UserEncoder`;
- $\mathbf{E}_i \in \mathbb{R}^d$ — обучаемое представление айтема.

Чем больше значение $r(u, i)$, тем более релевантным считается айтем $i$ для пользователя $u$.

На этапе инференса айтемы ранжируются по убыванию релевантности, и модель возвращает top-K кандидатов.

В этом задании мы обучаем модель на задачу экстремальной многоклассовой классификации.

Для каждого пользователя в батче нужно предсказать один правильный айтем из всего каталога айтемов размера $|\mathcal{I}|$.

#### Формула

Пусть $i^+$ — следующий айтем для пользователя $u$, тогда:

$$
\mathcal{L}_{\text{softmax}} = - \sum_{u \in \mathbf{U}} \log p(i^+ \mid u) = - \sum_{u \in \mathbf{U}} \left[r(u, i^+) - \log \sum_{j\in\mathcal{I}} \exp(r(u,j))\right].
$$


#### Почему это лучший способ обучать модели для этой стадии

Full softmax использует информацию обо всём каталоге: обучение модели эквивалентно применению: мы ищем позитив из всего каталога на обучении, мы берем top-K айтемов из всего каталога на применении.

In [ ]:
class SoftmaxModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    item_embeddings = self.encoder.item_embeddings.weight
    logits = user_repr @ item_embeddings.T

    labels = inputs["label"]

    return F.cross_entropy(logits, labels)

In [ ]:
tests.test_softmax_model(SoftmaxModel)

All good! :)


In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_full = SoftmaxModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_full = torch.optim.Adam(params=model_full.parameters(), lr=LEARNING_RATE)
best_checkpoint_full  = train_model(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_full,
    optimizer=optimizer_full,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 1 of 1: 100%|██████████| 3747/3747 [08:06<00:00,  7.70it/s]



Epoch 1/1 - Average Train Loss: 9.8966


Evaluation: 100%|██████████| 19/19 [00:01<00:00, 10.61it/s]


Metrics:
  Hitrate@100: 0.3288
  Recall@100:  0.1064
  nDCG@100:    0.0386
  Coverage@100: 0.4515
--------------------------------------------------------------------------------
Training finished


In [ ]:
model_full.load_state_dict(best_checkpoint_full)
final_metrics_full = evaluation(
    yambda_eval_dataloader,
    model_full,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_recs(final_metrics_full)

Evaluation: 100%|██████████| 19/19 [00:01<00:00, 10.24it/s]


All good! :)


## 5. BCE loss (1 балл)

Главная проблема предыдущего подхода — вычисления и память: полный softmax обычно применим, когда каталог не слишком большой — примерно до десятков/сотен тысяч айтемов. Для каталогов в миллионы обычно используют более простые подходы. Пойдет по их усложнению. Самый простой из них Binary Cross-Entropy (BCE).

Для каждого пользователя $u$ мы рассматриваем:

- позитивный пример: айтем $i^+$, с которым пользователь действительно взаимодействовал (следующий после истории пользователя);
- негативные примеры: айтемы $i^-$, сэмплированные из каталога (обычно равномерно), с которыми пользователь не взаимодействовал.

Модель обучается предсказывать вероятность того, что айтем является релевантным для пользователя в данный момент времени.

#### Формула

Для одного пользователя $u$, позитивного айтема $i^+$ и множества негативных айтемов $\mathcal{I}^-$ функция потерь имеет вид:

$$
\mathcal{L}_{\text{BCE}} =
- \Big[
\log \sigma\bigl(r(u, i^+)\bigr)
+ \sum_{i^- \in \mathcal{I}^-}
\log \bigl(1 - \sigma(r(u, i^-))\bigr)
\Big]
$$

(!) Важно, в этом задании необходимо брать только один негатив для каждого позитива (!)

In [ ]:
class BCEModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    bs = user_repr.shape[0]
    item_embeddings = self.encoder.item_embeddings.weight

    pos_emb = item_embeddings[inputs["label"]]

    neg_items = torch.randint(0, item_embeddings.shape[0], (bs,), device=user_repr.device)
    neg_emb = item_embeddings[neg_items]

    pos_scores = (user_repr * pos_emb).sum(dim=1)
    neg_scores = (user_repr * neg_emb).sum(dim=1)

    pos_loss = -F.logsigmoid(pos_scores)
    neg_loss = -F.logsigmoid(-neg_scores)

    return (pos_loss + neg_loss).mean()


In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_bce = BCEModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bce = torch.optim.Adam(params=model_bce.parameters(), lr=LEARNING_RATE)
best_checkpoint_bce = train_model(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bce,
    optimizer=optimizer_bce,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 1 of 1: 100%|██████████| 3747/3747 [02:39<00:00, 23.49it/s]



Epoch 1/1 - Average Train Loss: 0.9110


Evaluation: 100%|██████████| 19/19 [00:01<00:00, 12.06it/s]


Metrics:
  Hitrate@100: 0.1826
  Recall@100:  0.0480
  nDCG@100:    0.0165
  Coverage@100: 0.2410
--------------------------------------------------------------------------------
Training finished


In [ ]:
model_bce.load_state_dict(best_checkpoint_bce)
final_metrics_bce = evaluation(
    yambda_eval_dataloader,
    model_bce,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bce_recs(final_metrics_bce)

Evaluation: 100%|██████████| 19/19 [00:01<00:00,  9.65it/s]


All good! :)


## 6. BPR loss (1 балл)

Помимо BCE, двухбашенные модели также могут обучаться с использованием Bayesian Personalized Ranking (BPR).

Модель обучается на парах айтемов:

- позитивный айтем $i^+$, с которым пользователь действительно взаимодействовал;
- негативный айтем $i^-$, с которым пользователь не взаимодействовал (в данном подходе выбирается равномерно из каталога).

Цель обучения — добиться, чтобы для каждого пользователя выполнялось:

$$
r(u, i^+) > r(u, i^-)
$$

Таким образом, BPR напрямую приближает оптимизацию метрик ранжирования (Recall@K, nDCG@K), что делает его подходящим для retrieval-моделей.

#### Формула

Для каждого пользователя $u$ выбирается один позитивный айтем $i^+$ и один негативный айтем $i^-$. Функция потерь BPR определяется как:

$$
\mathcal{L}_{\text{BPR}}
= - \sum_{u \in \mathbf{U}} \log \sigma \bigl(r(u, i^+) - r(u, i^-)\bigr),
$$

где:
- $\mathbf{U}$ - набор польователей в батче;
- $r(u, i)$ — скор релевантности пользователя $u$ и айтема $i$;
- $\sigma(x) = \frac{1}{1 + e^{-x}}$ — сигмоида.

(!) Важно, в этом задании необходимо брать только один негатив для каждого позитива (!)

In [ ]:
class BPRModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    bs = user_repr.shape[0]

    item_embeddings = self.encoder.item_embeddings.weight

    pos_emb = item_embeddings[inputs["label"]]

    neg_emb = item_embeddings[torch.randint(0, item_embeddings.shape[0], (bs,), device=user_repr.device)]

    pos_scores = (user_repr * pos_emb).sum(dim=1)
    neg_scores = (user_repr * neg_emb).sum(dim=1)

    return -F.logsigmoid(pos_scores - neg_scores).mean()

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_bpr = BPRModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bpr = torch.optim.Adam(params=model_bpr.parameters(), lr=LEARNING_RATE)
best_checkpoint_bpr = train_model(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bpr,
    optimizer=optimizer_bpr,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 1 of 1: 100%|██████████| 3747/3747 [02:41<00:00, 23.27it/s]



Epoch 1/1 - Average Train Loss: 0.2536


Evaluation: 100%|██████████| 19/19 [00:02<00:00,  8.15it/s]


Metrics:
  Hitrate@100: 0.2297
  Recall@100:  0.0642
  nDCG@100:    0.0222
  Coverage@100: 0.1469
--------------------------------------------------------------------------------
Training finished


In [ ]:
model_bpr.load_state_dict(best_checkpoint_bpr)
final_metrics_bpr = evaluation(
    yambda_eval_dataloader,
    model_bpr,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bpr_recs(final_metrics_bpr)

Evaluation: 100%|██████████| 19/19 [00:01<00:00, 12.32it/s]


All good! :)


## 7. Sampled softmax, uniform negatives (1 балл)

Sampled softmax — это аппроксимация полного softmax: вместо всех айтемов мы берём небольшой их набор и считаем softmax только по нему.

Для каждого пользователя $u$ у нас есть:
- позитивный айтем $i^+$;
- множество семплированных негативов $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$.

#### Формула

$$
\mathcal{L}_{\text{sampled-uniform}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

In [ ]:
class SampledUniformModel(TwoTower):
  def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.init_weights(0.02)

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    bs = user_repr.shape[0]

    item_embeddings = self.encoder.item_embeddings.weight
    num_items = item_embeddings.shape[0]

    pos_labels = inputs["label"]
    pos_emb = item_embeddings[pos_labels]
    pos_scores = (user_repr * pos_emb).sum(dim=1, keepdim=True)

    neg_idx = torch.randint(0, num_items, (bs, self.num_negatives), device=user_repr.device)
    neg_emb = item_embeddings[neg_idx]
    neg_scores = (user_repr.unsqueeze(1) * neg_emb).sum(dim=2)

    logits = torch.cat([pos_scores, neg_scores], dim=1)
    labels = torch.zeros(bs, dtype=torch.long, device=user_repr.device)

    return F.cross_entropy(logits, labels)


In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_uniform = SampledUniformModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_uniform = torch.optim.Adam(params=model_sampled_uniform.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_uniform = train_model(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_uniform,
    optimizer=optimizer_sampled_uniform,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 1 of 1: 100%|██████████| 3747/3747 [06:21<00:00,  9.81it/s]



Epoch 1/1 - Average Train Loss: 5.5636


Evaluation: 100%|██████████| 19/19 [00:01<00:00,  9.83it/s]


Metrics:
  Hitrate@100: 0.3254
  Recall@100:  0.1053
  nDCG@100:    0.0380
  Coverage@100: 0.4387
--------------------------------------------------------------------------------
Training finished


In [ ]:
model_sampled_uniform.load_state_dict(best_checkpoint_sampled_uniform)
final_metrics_sampled_uniform = evaluation(
    yambda_eval_dataloader,
    model_sampled_uniform,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_uniform_recs(final_metrics_sampled_uniform)

Evaluation: 100%|██████████| 19/19 [00:01<00:00, 12.24it/s]


All good! :)


## 8. Sampled softmax, in-batch negatives (1 балл)

В прошлой задаче мы приближали полный softmax, сэмплируя негативы равновероятно из каталога. Теперь рассмотрим ещё более популярный подход: использование in-batch негативов.

Для каждого пользователя $u$ у нас есть:
- позитивный айтем $i^+$;
- множество семплированных негативов $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$ (только теперь мы семплируем не из всего каталога, а из батча).

#### Формула

$$
\mathcal{L}_{\text{sampled-batch}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

(!) Важно, в этом задании негативы необходимо семллировать не только ли `label`, но и из `history` (!)

In [ ]:
class SampledInBatchModel(TwoTower):
    def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
      super().__init__(num_items=num_items, embedding_dim=embedding_dim)
      self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
      self.num_negatives = num_negatives
      self.temperature = 0.1
      self.l2_lambda = 1e-5
      self.init_weights(0.02)

    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
      bs = user_repr.shape[0]
      item_embeddings = self.encoder.item_embeddings.weight

      pos_emb = item_embeddings[inputs["label"]]
      pos_scores = (user_repr * pos_emb).sum(dim=1, keepdim=True)

      in_batch_items = torch.cat([
        inputs["label"],
        inputs["history"]["item_id"]
      ])
      neg_items = torch.unique(in_batch_items)

      neg_emb = item_embeddings[neg_items]
      neg_scores = user_repr @ neg_emb.T

      logits = torch.cat([pos_scores, neg_scores], dim=1)

      logits = logits / self.temperature

      labels = torch.zeros(bs, dtype=torch.long, device=user_repr.device)

      ce_loss = F.cross_entropy(logits, labels)

      l2_reg = (user_repr ** 2).mean() + (pos_emb ** 2).mean()

      return ce_loss + self.l2_lambda * l2_reg

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_in_batch = SampledInBatchModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_in_batch = torch.optim.Adam(params=model_sampled_in_batch.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch = train_model(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch,
    optimizer=optimizer_sampled_in_batch,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 1 of 1: 100%|██████████| 3747/3747 [05:18<00:00, 11.78it/s]



Epoch 1/1 - Average Train Loss: 9.6897


Evaluation: 100%|██████████| 19/19 [00:01<00:00, 13.11it/s]


Metrics:
  Hitrate@100: 0.2944
  Recall@100:  0.0905
  nDCG@100:    0.0327
  Coverage@100: 0.5948
--------------------------------------------------------------------------------
Training finished


In [ ]:
model_sampled_in_batch.load_state_dict(best_checkpoint_sampled_in_batch)
final_metrics_sampled_in_batch = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_recs(final_metrics_sampled_in_batch)

Evaluation: 100%|██████████| 19/19 [00:01<00:00, 10.26it/s]


All good! :)


## 9. Sampled softmax, in-batch negatives + logq correction (1 балл)

Использование in-batch подход это быстро и эффективно, но остаётся важная проблема: такое распределение негативов не совпадает с распределением в случае полного или uniform sampled softmax.

Один из стандартных способов избавиться от смещения — добавить log-q коррекцию.

#### Почему нужна коррекция

Обычный in-batch подход воспринимает все негативы как “равноправные”, но в реальности некоторые айтемы встречаются гораздо чаще, другие — почти никогда.

То есть негативы получаются как выборка из некоторого распределения $q(i)$, а не равномерные. Если мы хотим приблизиться к полному softmax, нужно компенсировать это смещение.

#### Формула

Пусть $q(i)$ — вероятность того, что айтем $i$ будет появляться как негатив-кандидат.

Тогда корректируем логит негативов:

$$
\tilde{r}(u,i) = r(u,i) - \log q(i),
$$

где $q(i)$ — вероятность появления айтема $i$ в качетстве негатива: частота его появления среди всех позитивов: $\frac{\#i}{\#all}$.

In [ ]:
def build_q_from_train_interactions(
  train_data: pl.DataFrame,
  catalog_size: int,
  item_col: str = "item_id",
  eps: float = 1e-12,
) -> torch.Tensor:
  item_counts = train_data.group_by(item_col).agg(pl.len().alias("count"))

  q = torch.zeros(catalog_size, dtype=torch.float32)
  counts = item_counts["count"].to_numpy()

  q[item_counts[item_col].to_numpy()] = torch.tensor(counts, dtype=torch.float32)
  return q + eps

In [ ]:
class SampledInBatchModelLogQ(TwoTower):
  def __init__(
    self,
    num_items: int,
    embedding_dim: int,
    num_negatives: int,
    q: torch.Tensor,
    eps: float = 1e-12,
  ) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.eps = eps
    self.temperature = 0.1

    q = q.detach().float()
    q = q / (q.sum() + eps)
    logq = torch.log(q.clamp_min(eps))
    self.register_buffer("logq", logq)

    self.init_weights(0.02)

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    bs = user_repr.shape[0]
    item_embeddings = self.encoder.item_embeddings.weight

    pos_labels = inputs["label"]
    pos_emb = item_embeddings[pos_labels]
    pos_scores = (user_repr * pos_emb).sum(dim=1, keepdim=True)
    pos_logq = self.logq[pos_labels].unsqueeze(1)
    pos_scores_corrected = pos_scores - pos_logq

    in_batch_items = torch.unique(torch.cat([
        pos_labels,
        inputs["history"]["item_id"]
    ]))

    neg_emb = item_embeddings[in_batch_items]
    neg_scores = user_repr @ neg_emb.T
    neg_logq = self.logq[in_batch_items].unsqueeze(0)
    neg_scores_corrected = neg_scores - neg_logq

    logits = torch.cat([pos_scores_corrected, neg_scores_corrected], dim=1)
    labels = torch.zeros(bs, dtype=torch.long, device=user_repr.device)

    return F.cross_entropy(logits, labels)



In [ ]:
gc.collect()
torch.cuda.empty_cache()

q = build_q_from_train_interactions(
    train_data=train,
    catalog_size=catalog_size,
    item_col="item_idx"
)

model_sampled_in_batch_logq = SampledInBatchModelLogQ(num_items=catalog_size, embedding_dim=64, num_negatives=2048, q=q).to(DEVICE)
optimizer_sampled_in_batch_logq = torch.optim.Adam(params=model_sampled_in_batch_logq.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch_logq = train_model(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch_logq,
    optimizer=optimizer_sampled_in_batch_logq,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 1 of 1: 100%|██████████| 3747/3747 [05:05<00:00, 12.27it/s]



Epoch 1/1 - Average Train Loss: 9.9938


Evaluation: 100%|██████████| 19/19 [00:01<00:00, 12.93it/s]


Metrics:
  Hitrate@100: 0.3211
  Recall@100:  0.1023
  nDCG@100:    0.0373
  Coverage@100: 0.3305
--------------------------------------------------------------------------------
Training finished


In [ ]:
model_sampled_in_batch_logq.load_state_dict(best_checkpoint_sampled_in_batch_logq)
final_metrics_sampled_in_batch_logq = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch_logq,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_logq_recs(final_metrics_sampled_in_batch_logq)

Evaluation: 100%|██████████| 19/19 [00:02<00:00,  8.31it/s]


All good! :)


# Лидерборд и выводы

Собираем таблицу со всеми методами и метриками.

In [ ]:
leaderboard = pl.DataFrame([
    {"method": "Softmax loss", **final_metrics_full},
    {"method": "BCE loss", **final_metrics_bce},
    {"method": "BPR loss", **final_metrics_bpr},
    {"method": "Sampled softmax, uniform negatives", **final_metrics_sampled_uniform},
    {"method": "Sampled softmax, in-batch negatives", **final_metrics_sampled_in_batch},
    {"method": "Sampled softmax, in-batch negatives + logq correction", **final_metrics_sampled_in_batch_logq},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard

method,hitrate,ndcg,coverage,recall
str,f64,f64,f64,f64
"""Softmax loss""",0.32882,0.038557,0.451529,0.106351
"""Sampled softmax, uniform negat…",0.325375,0.038038,0.43874,0.105349
"""Sampled softmax, in-batch nega…",0.321102,0.037305,0.330459,0.102292
"""Sampled softmax, in-batch nega…",0.294397,0.032715,0.594806,0.090508
"""BPR loss""",0.229691,0.022213,0.146885,0.064226
"""BCE loss""",0.182583,0.01647,0.241001,0.047957


## 10. Вопросы на понимание (1 балл)

1. В чем основная проблема использования `Full softmax`?
2. Почему `BCE` хуже показал себя чем `BPR`?
3. В чем может быть проблема с `Sampled softmax, uniform` подходом?
4. В чем проблема `in-batch` подхода без использования `logq`-коррекции?
5. Почему при добавлении `logq` у нас упал `coverage`?

Ответы - текстом

In [ ]:
#ответы



1. Основная проблема - вычислительная сложность. Считать скоры для всех айтемов католога затратно.

2. BCE - улучшает класиффикацию, BPR - ранжирование. В BCE модель предсказывает "релевантность", BPR - оптимизирует попарное ранжирование, что "нужнее" для рекомендации

3. При uniform sampling каждый айтем с равной вероятностью становится негативом, не учитывается реальное распределение айтемов. Много сэмплов тратится на "лёгкие" негативы.

4. Самая главная проблема - смещение в сторону популярных айтемов. Популярные айтемы чаще бывают в батчках как позитивы других пользователей и модель их "задавливает"

5. Редкие айтемы с маленьким q получают большую надбавку => модель начинает рекомендовать узкий набор таких айтемов с "завышенной" оценкой



# Бонусные задания

Вы уже реализовали основные подходы, провели замеры и сделали выводы о том, как разные функции потерь и стратегии негативного сэмплирования влияют на качество модели. В качестве бонусных заданий предлагается реализовать более продвинутые методы, об одном которых мы также говорили на лекции.

## 11. Sampled softmax, in-batch negatives + **fixed** logq correction (1 балл)

В этом бонусном задании реализуем более аккуратный вариант `logq correction`, предложенный в статье *Correcting the LogQ Correction: Revisiting Sampled Softmax for Large-Scale Retrieval*. Стандартная `logq`-коррекция не полностью устраняет смещение, возникающее из-за неравномерного появления объектов в батче.

Ключевая идея состоит в том, что в стандартном выводе `logq` положительный объект неявно трактуется так, будто он был получен из того же распределения, что и негативы. На практике это не так: положительный объект всегда присутствует в примере детерминированно и не является случайно выбранным негативом. Именно эта деталь приводит к дополнительному смещению.

Реализуйте **fixed logq correction**: исправленный вариант коррекции, который учитывает, что политивный пример не должен обрабатываться так же, как семплированные негативы.

In [ ]:
class SampledInBatchModelFixedLogQ(TwoTower):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q: torch.Tensor,
        eps: float = 1e-12,
    ) -> None:
        super().__init__(num_items=num_items, embedding_dim=embedding_dim)
        self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
        self.num_negatives = num_negatives
        self.eps = eps

        q = q.detach().float()
        q = q / q.sum()
        self.register_buffer("q", q)

        self.init_weights(0.02)

    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_in_batch_logq_fixed = SampledInBatchModelFixedLogQ(num_items=catalog_size, embedding_dim=64, num_negatives=2048, q=q).to(DEVICE)
optimizer_sampled_in_batch_logq_fixed = torch.optim.Adam(params=model_sampled_in_batch_logq_fixed.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch_logq_fixed = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch_logq_fixed,
    optimizer=optimizer_sampled_in_batch_logq_fixed,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

In [ ]:
model_sampled_in_batch_logq_fixed.load_state_dict(best_checkpoint_sampled_in_batch_logq_fixed)
final_metrics_sampled_in_batch_logq_fixed = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch_logq_fixed,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_logq_fixed_recs(final_metrics_sampled_in_batch_logq_fixed)

## 12. Улучшение аггрегации история пользователя (1 балл)

В этом бонусном задании вам предлагается самостоятельно улучшить способ агрегации истории пользователя в модели и добиться дополнительного прироста качества. В базовых решениях история пользователя уже используется для построения пользовательского представления, однако сама схема агрегации может быть довольно простой и не всегда позволяет достаточно хорошо учитывать порядок, важность и контекст прошлых взаимодействий.

Цель этого задания — получить **дополнительный прирост качества не менее чем на 0.01 по nDCG в абсолютных значениях** по сравнению с вашим лучшим решением из предыдущих пунктов. Иными словами, если ваш лучший результат раньше был, например, `nDCG@K = 0.123`, то для выполнения этого бонусного задания нужно получить как минимум `0.133`.

Важно: этот пункт проверяющие **не проверяли заранее самостоятельно**, поэтому дополнительны балл будет выставляться только за тот код, который действительно является **воспроизводимым** в **Google Colab на GPU T4** и получить такие же или очень близкие результаты. Поэтому в решении особенно важно:
- зафиксировать сиды
- явно указать все изменения в модели
- сохранить корректный и полный пайплайн обучения
- не опускать важные ячейки с подготовкой данных, обучением и оценкой


In [ ]:
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################
final_metrics_your_solution = ...
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

## Лидерборд с бонусами

In [ ]:
leaderboard = pl.DataFrame([
    {"method": "Softmax loss", **final_metrics_full},
    {"method": "BCE loss", **final_metrics_bce},
    {"method": "BPR loss", **final_metrics_bpr},
    {"method": "Sampled, uniform", **final_metrics_sampled_uniform},
    {"method": "Sampled, in-batch", **final_metrics_sampled_in_batch},
    {"method": "Sampled, in-batch + logq", **final_metrics_sampled_in_batch_logq},
    {"method": "Sampled, in-batch + fixed logq", **final_metrics_sampled_in_batch_logq_fixed},
    {"method": "Your custom solution", **final_metrics_your_solution},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard